# Master Table Data Profiling

## Description
This notebook profiles the master tables in the data warehouse to assess data quality and identify potential issues. The profiling covers data types, missing values, duplicate records, categorical distributions, and other data quality concerns.

## Master Tables Profiled
- Branch
- Product
- Inventory
- Supplier
- Customer

## Objective
To identify potential data inconsistencies and recommend improvements to ensure data accuracy, consistency, and reliability.


In [1]:
import duckdb  
import pandas as pd 

con = duckdb.connect("warehouse.duckdb")

In [2]:
con.execute("SHOW TABLES").df()

,name
0,branches
1,customers
2,inventory_master
3,invoices
4,payments
5,products
6,purchase_orders_header
7,purchase_orders_lines
8,sales_orders_header
9,sales_orders_lines


In [7]:
# Profiling master tables: branches, products, inventory_master, suppliers, customers
# Create a profiling function   
def profile_table(con, table_name): 
    schema = con.execute( 
        f'DESCRIBE "{table_name}"' 
    ).df() 
 
    row_count = con.execute( 
        f'SELECT COUNT(*) FROM "{table_name}"' 
    ).fetchone()[0] 
 
    duplicate_rows = con.execute(f""" 
        SELECT COUNT(*) 
        FROM ( 
            SELECT * 
            FROM "{table_name}" 
            GROUP BY ALL 
            HAVING COUNT(*) > 1 
        ) AS duplicates 
    """).fetchone()[0] 
 
    missing_values = {} 
 
    for column in schema["column_name"]: 
        query = f''' 
 
            SELECT COUNT(*) - COUNT("{column}") 
            FROM "{table_name}" 
        ''' 
 
        missing_values[column] = con.execute( 
            query 
        ).fetchone()[0] 
 
    return { 
        "table_name": table_name, 
        "row_count": row_count, 
        "column_count": len(schema), 
        "schema": schema, 
        "missing_values": missing_values, 
        "duplicate_rows": duplicate_rows 
    }

In [8]:
master_tables = [ 
    "branches", 
    "products", 
    "inventory_master",
    "suppliers",
    "customers"
] 
 
for table in master_tables: 
    result = profile_table(con, table) 
 
    print("\nTable:", result["table_name"]) 
    print("Rows:", result["row_count"]) 
    print("Columns:", result["column_count"]) 
    print("Missing values:", result["missing_values"]) 
    print("Duplicate row patterns:", 
result["duplicate_rows"]) 
 
    display(result["schema"])


Table: branches
Rows: 6
Columns: 13
Missing values: {'branch_id': 0, 'branch_name': 0, 'city': 0, 'state': 0, 'region': 0, 'warehouse_type': 0, 'warehouse_capacity': 0, 'service_center_available': 0, 'manager_id': 0, 'total_employees': 0, 'avg_monthly_revenue': 0, 'monthly_operational_cost': 0, 'market_demand_index': 0}
Duplicate row patterns: 0


,column_name,column_type,null,key,default,extra
0,branch_id,VARCHAR,YES,None,None,None
1,branch_name,VARCHAR,YES,None,None,None
2,city,VARCHAR,YES,None,None,None
3,state,VARCHAR,YES,None,None,None
4,region,VARCHAR,YES,None,None,None
5,warehouse_type,VARCHAR,YES,None,None,None
6,warehouse_capacity,VARCHAR,YES,None,None,None
7,service_center_available,BOOLEAN,YES,None,None,None
8,manager_id,BIGINT,YES,None,None,None
9,total_employees,BIGINT,YES,None,None,None



Table: products
Rows: 30
Columns: 23
Missing values: {'product_id': 0, 'product_name': 0, 'category': 0, 'machine_type': 0, 'brand': 0, 'model_compatibility': 0, 'unit_cost': 0, 'unit_price': 0, 'margin_percentage': 0, 'gst_rate': 0, 'weight_kg': 0, 'dimensions_cm': 0, 'material_type': 0, 'warranty_months': 0, 'reorder_level': 0, 'safety_stock': 0, 'max_stock_level': 0, 'lead_time_days': 0, 'criticality_level': 0, 'usage_frequency': 0, 'uom': 0, 'last_purchase_price': 0, 'last_purchase_date': 0}
Duplicate row patterns: 0


,column_name,column_type,null,key,default,extra
0,product_id,VARCHAR,YES,None,None,None
1,product_name,VARCHAR,YES,None,None,None
2,category,VARCHAR,YES,None,None,None
3,machine_type,VARCHAR,YES,None,None,None
4,brand,VARCHAR,YES,None,None,None
5,model_compatibility,VARCHAR,YES,None,None,None
6,unit_cost,BIGINT,YES,None,None,None
7,unit_price,BIGINT,YES,None,None,None
8,margin_percentage,DOUBLE,YES,None,None,None
9,gst_rate,BIGINT,YES,None,None,None



Table: inventory_master
Rows: 180
Columns: 8
Missing values: {'product_id': 0, 'branch_id': 0, 'opening_stock': 0, 'reorder_level': 0, 'safety_stock': 0, 'max_stock': 0, 'current_stock': 0, 'warehouse_bin': 0}
Duplicate row patterns: 0


,column_name,column_type,null,key,default,extra
0,product_id,VARCHAR,YES,None,None,None
1,branch_id,VARCHAR,YES,None,None,None
2,opening_stock,BIGINT,YES,None,None,None
3,reorder_level,BIGINT,YES,None,None,None
4,safety_stock,BIGINT,YES,None,None,None
5,max_stock,BIGINT,YES,None,None,None
6,current_stock,BIGINT,YES,None,None,None
7,warehouse_bin,VARCHAR,YES,None,None,None



Table: suppliers
Rows: 8
Columns: 12
Missing values: {'supplier_id': 0, 'supplier_name': 0, 'supplier_type': 0, 'product_category': 0, 'city': 0, 'province': 0, 'region': 0, 'pincode': 0, 'lead_time_days': 0, 'reliability_score': 0, 'import_duty_rate': 0, 'china_tax_id': 0}
Duplicate row patterns: 0


,column_name,column_type,null,key,default,extra
0,supplier_id,VARCHAR,YES,None,None,None
1,supplier_name,VARCHAR,YES,None,None,None
2,supplier_type,VARCHAR,YES,None,None,None
3,product_category,VARCHAR,YES,None,None,None
4,city,VARCHAR,YES,None,None,None
5,province,VARCHAR,YES,None,None,None
6,region,VARCHAR,YES,None,None,None
7,pincode,BIGINT,YES,None,None,None
8,lead_time_days,BIGINT,YES,None,None,None
9,reliability_score,BIGINT,YES,None,None,None



Table: customers
Rows: 500
Columns: 15
Missing values: {'customer_id': 0, 'customer_type': 0, 'industry_segment': 0, 'city': 0, 'state': 0, 'pincode': 0, 'region': 0, 'branch_id': 0, 'credit_limit': 0, 'current_balance': 0, 'payment_terms': 0, 'customer_since': 0, 'last_purchase_date': 0, 'total_purchase_value': 0, 'customer_rating': 0}
Duplicate row patterns: 0


,column_name,column_type,null,key,default,extra
0,customer_id,VARCHAR,YES,None,None,None
1,customer_type,VARCHAR,YES,None,None,None
2,industry_segment,VARCHAR,YES,None,None,None
3,city,VARCHAR,YES,None,None,None
4,state,VARCHAR,YES,None,None,None
5,pincode,BIGINT,YES,None,None,None
6,region,VARCHAR,YES,None,None,None
7,branch_id,VARCHAR,YES,None,None,None
8,credit_limit,BIGINT,YES,None,None,None
9,current_balance,BIGINT,YES,None,None,None


In [38]:
# Top 5 rows
con.execute("""
SELECT *
FROM branches
LIMIT 5
""").df()

,branch_id,branch_name,city,state,region,warehouse_type,warehouse_capacity,service_center_available,manager_id,total_employees,avg_monthly_revenue,monthly_operational_cost,market_demand_index
0,DEL001,Delhi Central,Delhi,Delhi,North,Central,45230 sqft,True,9593,92,5234890,3189000,9
1,PUN001,Pune Distribution,Pune,Maharashtra,West,Regional,38750 sqft,True,6114,74,4389120,2654000,8
2,CHN001,Chennai South Hub,Chennai,Tamil Nadu,South,Regional,41890 sqft,True,7922,81,4567780,2739000,8
3,HYD001,Hyderabad Logistics,Hyderabad,Telangana,South,Branch,36210 sqft,False,4390,57,3102455,1887000,7
4,KOL001,Kolkata East Depot,Kolkata,West Bengal,East,Branch,29880 sqft,False,5981,49,2894770,1743000,6


In [39]:
# Top 5 rows
con.execute("""
SELECT *
FROM products 
LIMIT 5
""").df()

,product_id,product_name,category,machine_type,brand,model_compatibility,unit_cost,unit_price,margin_percentage,gst_rate,...,warranty_months,reorder_level,safety_stock,max_stock_level,lead_time_days,criticality_level,usage_frequency,uom,last_purchase_price,last_purchase_date
0,P001,Hydraulic Pump HP-300,Hydraulic,Excavator,CAT,CAT 320D,18950,24500,29.2,28,...,12,15,10,40,21,High,Medium,piece,18500,2019-03-15
1,P002,Engine Oil Filter OF-90,Filters,Loader,JCB,JCB 3DX,450,720,37.3,18,...,6,80,40,200,7,Medium,High,piece,430,2019-07-22
2,P003,Air Filter AF-120,Filters,Excavator,Komatsu,Komatsu PC210,820,1250,52.4,18,...,6,60,30,150,7,Medium,High,piece,800,2020-01-10
3,P004,Track Chain TC-45,Undercarriage,Bulldozer,Volvo,Volvo D6,32500,39800,22.5,28,...,18,8,4,20,30,High,Low,set,32000,2020-06-18
4,P005,Fuel Injector FI-220,Engine,Excavator,Tata Hitachi,Tata Hitachi EX 200,6350,8450,33.0,28,...,12,30,15,80,18,High,Medium,piece,6200,2020-11-05


In [40]:
# Top 5 rows
con.execute("""
SELECT *
FROM inventory_master 
LIMIT 5
""").df()

,product_id,branch_id,opening_stock,reorder_level,safety_stock,max_stock,current_stock,warehouse_bin
0,P001,DEL001,196,59,46,340,115928,C15
1,P001,PUN001,198,70,43,349,109389,D16
2,P001,CHN001,187,62,44,311,99373,A11
3,P001,HYD001,226,71,65,417,108772,A10
4,P001,KOL001,263,94,58,475,103808,F03


In [41]:
# Top 5 rows
con.execute("""
SELECT *
FROM suppliers 
LIMIT 5
""").df()

,supplier_id,supplier_name,supplier_type,product_category,city,province,region,pincode,lead_time_days,reliability_score,import_duty_rate,china_tax_id
0,SUP0001,Shenzhen OEM Supplies,OEM,Medium Parts,Shenzhen,Guangdong,South China,586808,19,5,8,90SUZHOU398241
1,SUP0002,Guangzhou Local Vendor Supplies,Local Vendor,Small Parts,Guangzhou,Guangdong,South China,966203,10,5,12,89TIANJI357980
2,SUP0003,Shanghai Distributor Supplies,Distributor,Large Parts,Shanghai,Shanghai,East China,655757,16,4,8,96SHENZH983302
3,SUP0004,Ningbo Distributor Supplies,Distributor,Large Parts,Ningbo,Zhejiang,East China,536477,18,3,8,39GUANGZ946640
4,SUP0005,Tianjin OEM Supplies,OEM,Small Parts,Tianjin,Tianjin,North China,334261,28,3,5,59GUANGZ986793


In [42]:
# Top 5 rows
con.execute("""
SELECT *
FROM customers 
LIMIT 5
""").df()

,customer_id,customer_type,industry_segment,city,state,pincode,region,branch_id,credit_limit,current_balance,payment_terms,customer_since,last_purchase_date,total_purchase_value,customer_rating
0,C0001,Retail,Construction,Kolkata,West Bengal,991035,East,KOL001,45896,43284,Advance,2024-04-10,2024-04-25,3892818,4
1,C0002,Fleet Owner,Manufacturing,Hyderabad,Telangana,464589,South,HYD001,531506,426735,Advance,2022-01-19,2024-12-04,3168730,2
2,C0003,Government,Logistics,Surat,Gujarat,202475,West,DEL001,1306933,733495,Net 30,2017-02-14,2019-06-11,3584708,2
3,C0004,Fleet Owner,Logistics,Chennai,Tamil Nadu,576213,South,CHN001,506336,250255,Net 30,2017-12-20,2021-12-26,3810635,3
4,C0005,Dealer,Construction,Ahmedabad,Gujarat,965661,West,AHM001,380783,183927,Net 30,2023-07-27,2024-02-17,1381374,2


In [34]:
## Category Profiling for branches
con.execute("""
SELECT 'States' AS category, COUNT(DISTINCT state) AS total
FROM branches

UNION ALL

SELECT 'Cities' AS category, COUNT(DISTINCT city) AS total
FROM branches

UNION ALL

SELECT 'Regions' AS category, COUNT(DISTINCT region) AS total
FROM branches

UNION ALL

SELECT 'Warehouse Types' AS category, COUNT(DISTINCT warehouse_type) AS total
FROM branches;
""").df()

,category,total
0,States,6
1,Cities,6
2,Regions,4
3,Warehouse Types,4


In [32]:
## Category Profiling for products
con.execute("""
SELECT 'category' AS category, COUNT(DISTINCT category) AS total
FROM products

UNION ALL

SELECT 'machine_type' AS category, COUNT(DISTINCT machine_type) AS total
FROM products

UNION ALL

SELECT 'brand' AS category, COUNT(DISTINCT brand) AS total
FROM products

UNION ALL

SELECT 'criticality_level' AS category, COUNT(DISTINCT criticality_level) AS total
FROM products

UNION ALL

SELECT 'material_type' AS category, COUNT(DISTINCT material_type) AS total
FROM products;
""").df()

,category,total
0,category,15
1,machine_type,6
2,brand,6
3,criticality_level,3
4,material_type,10


In [33]:
## Category Profiling for inventory_master
con.execute("""
SELECT 'warehouse_bin' AS category, COUNT(DISTINCT warehouse_bin) AS total
FROM inventory_master

""").df()

,category,total
0,warehouse_bin,95


In [31]:
## Category Profiling for suppliers
con.execute("""
SELECT 'supplier_type' AS category, COUNT(DISTINCT supplier_type) AS total
FROM suppliers

UNION ALL

SELECT 'product_category' AS category, COUNT(DISTINCT product_category) AS total
FROM suppliers

UNION ALL

SELECT 'city' AS category, COUNT(DISTINCT city) AS total
FROM suppliers

UNION ALL

SELECT 'province' AS category, COUNT(DISTINCT province) AS total
FROM suppliers

UNION ALL

SELECT 'region' AS category, COUNT(DISTINCT region) AS total
FROM suppliers;
""").df()

,category,total
0,supplier_type,3
1,product_category,3
2,city,8
3,province,6
4,region,3


In [30]:
## Category Profiling for customers
con.execute("""
SELECT 'city' AS category, COUNT(DISTINCT city) AS total
FROM customers

UNION ALL

SELECT 'state' AS category, COUNT(DISTINCT state) AS total
FROM customers

UNION ALL

SELECT 'region' AS category, COUNT(DISTINCT region) AS total
FROM customers

UNION ALL

SELECT 'payment_terms' AS category, COUNT(DISTINCT payment_terms) AS total
FROM customers;
""").df()

,category,total
0,city,21
1,state,16
2,region,6
3,payment_terms,5


# Branches Table Profiling Findings
* Total rows: 6 
* Total columns: 13
* Missing values: 0
* Duplicate rows: 0
###  Data Type Observations
- **warehousecapacity:** Stored as `VARCHAR`, possibly because the values include measurement units such as `sqft`. Confirm the actual values before determining whether conversion to a numeric data type is necessary.
- **branchid:** Stored as `VARCHAR`, which is appropriate because the identifiers contain a mixture of letters and numbers.
- **managerid:** Stored as `BIGINT`, which is suitable for numeric identifiers.

###  Other Data Quality Concerns
- **Nullability:** The columns `branchid`, `branchname`, `city`, `state`, `region`, `managerid`, and `totalemployees` allow null values. Missing values could affect branch identification, location reporting, and workforce analysis. Confirm whether these columns should require values based on business rules.


## Products Table Profiling Findings
* Total rows: 30 
* Total columns: 23
* Missing values: 0
* Duplicate rows: 0
## 🔎 Data Type Observations

- **VARCHAR Usage:** Several fields, including `productid`, `productname`, `category`, `brand`, and `dimensionscm`, use VARCHAR. The `dimensionscm` field may benefit from structured numeric storage to support calculations and improve consistency.

- **BIGINT for Costs:** Fields such as `unitcost`, `unitprice`, and `lastpurchaseprice` use BIGINT. DECIMAL or NUMERIC may be more appropriate for monetary values requiring decimal precision.

- **DOUBLE Precision:** Fields such as `marginpercentage` and `weightkg` use DOUBLE. DECIMAL may be preferable where precise calculations are required.

- **DATE Field:** `lastpurchasedate` uses DATE, which is appropriate for recording purchase dates. TIMESTAMP may be needed if time information is required.

- **Categorical Fields:** `criticalitylevel` and `usagefrequency` use VARCHAR. Predefined categories or lookup tables could improve consistency.

## ⚠️ Data Quality Concerns

- **Primary Key Missing:** `product_id` is not marked as a primary key. Its uniqueness should be verified before establishing it as the primary key.

- **Null Allowance:** All columns allow NULL values. Critical fields such as `productid`, `productname`, and `unitprice` should be reviewed to determine whether NOT NULL constraints are necessary.

- **Default Values:** No default values are defined. Fields such as `reorderlevel`, `safetystock`, and `leadtime_days` could benefit from appropriate defaults where supported by business rules.

- **Data Consistency:** Free-text fields such as `dimensionscm` and `materialtype` may contain inconsistent formats or descriptions. Standardizing their formats would improve data quality.


## Inventory_Master Table Profiling Findings
* Total rows: 180 
* Total columns: 8
* Missing values: 0
* Duplicate rows: 0
##  Data Type Observations

- **Identifiers:** `productid` and `branchid` are stored as `VARCHAR`. This is appropriate for alphanumeric identifiers, provided their data types match those of the referenced columns.
- **Stock Columns:** `openingstock`, `reorderlevel`, `safetystock`, `maxstock`, and `current_stock` are stored as `BIGINT`. This supports large stock quantities, although `INT` may be sufficient if the expected values are smaller.

##  Data Quality Concerns

- **Nullability:** All columns are marked as nullable (`YES`). Critical identifiers should be reviewed to determine whether they require `NOT NULL` constraints.
- **Default Values:** No default values are defined. A default of `0` for `current_stock` may be appropriate if it aligns with business rules.
- **Referential Integrity:** `productid` and `branchid` serve as foreign keys linking inventory records to the corresponding product and branch tables. Their references should be checked to ensure that every value matches an existing record in the appropriate table.



## Suppliers Table Profiling Findings
* Total rows: 8 
* Total columns: 12
* Missing values: 0
* Duplicate rows: 0
### Data Type Observations

- **VARCHAR Usage:** Most fields, including `supplierid` and `chinataxid`, are defined as VARCHAR. This is appropriate for alphanumeric identifiers, but consistent formatting should be maintained.

- **BIGINT Fields:** Attributes such as `pincode`, `leadtimedays`, `reliabilityscore`, and `importdutyrate` are stored as BIGINT. Consider using more appropriate data types:
  - `pincode`: VARCHAR or CHAR to preserve leading zeros.
  - `leadtimedays` and `reliabilityscore`: INTEGER if they contain whole numbers within the supported range.
  - `importdutyrate`: DECIMAL to accommodate percentage values with decimal places.

### Data Quality Concerns

- **Nullability:** All columns allow NULL values. Critical fields, such as `supplierid` and `supplier_name`, should ideally be NOT NULL to prevent incomplete records.

- **Primary Key:** No primary key is defined. Establishing a primary key for `supplierid` would help ensure unique supplier records and improve data integrity.

- **Default Values:** No default values are defined. Appropriate defaults may be considered where business rules permit, rather than automatically assigning zero to missing values.

- **Data Consistency:** Fields such as `city`, `province`, and `region` use free-text VARCHAR values, which may result in inconsistent spelling or naming conventions. Standardized values would improve consistency.

- **Tax ID Validation:** `chinataxid` is stored as VARCHAR without an indicated validation rule. Format checks should be applied to ensure tax identification numbers follow the expected structure.


## Customers Table Profiling Findings
* Total rows: 500 
* Total columns: 15
* Missing values: 0
* Duplicate rows: 0
### Data Type Observations

- **Customer ID:** Defined as `VARCHAR`. This is suitable for alphanumeric identifiers. However, if IDs follow a fixed-length format, `CHAR` may improve consistency.
- **Pincode:** Stored as `BIGINT`. Since pincodes are identifiers rather than values for calculations, `VARCHAR` or `CHAR` is more appropriate to preserve leading zeros.
- **Financial and Rating Fields:** `creditlimit`, `currentbalance`, `totalpurchasevalue`, and `customerrating` are defined as `BIGINT`. For monetary values that require decimal precision, `DECIMAL` is more appropriate to avoid rounding errors.
- **Date Fields:** `customersince` and `lastpurchase_date` are correctly defined as `DATE`. If time information is required, `DATETIME` or `TIMESTAMP` may be considered.

###  Data Quality Concerns

- **Nullability:** All columns allow `NULL` values. Critical fields, such as `customerid`, `customertype`, and `customersince`, should ideally be defined as `NOT NULL` to prevent incomplete records.
- **Primary Key and Uniqueness:** No primary key or unique constraint is defined. `customerid` should ideally be the primary key to ensure each customer is uniquely identified.
- **Default Values:** No default values are specified. Appropriate defaults may be considered for fields such as `currentbalance` and `customerrating`, where zero is a valid business value.
- **Categorical Consistency:** Fields such as `customertype`, `industry_segment`, and `region` may contain inconsistent entries due to differences in spelling or capitalization. Standardized values or reference tables can help maintain consistency.
- **VARCHAR Lengths:** Using `VARCHAR` without specified length limits may affect storage efficiency and data consistency. Appropriate maximum lengths should be defined where supported and necessary.







# Overall Data Quality Assessment

## 1. Overall Observations

The profiling of the Branch, Product, Inventory, Supplier, and Customer master tables identified potential areas for improvement in the data warehouse schema. These include data type suitability, nullable columns, primary key and uniqueness constraints, categorical consistency, and referential integrity.

Some fields may require more appropriate data types to support accurate calculations and consistent data representation.

## 2. Main Data Quality Concerns

- **Data Type Suitability:** Some columns may use data types that are not ideal for their intended purposes. Examples include pincodes stored as `BIGINT`, monetary values stored as `BIGINT`, and measurement fields stored as `VARCHAR`.

- **Nullability:** Several columns, including critical identifiers and descriptive fields, allow `NULL` values. These columns should be reviewed against business requirements.

- **Primary Key and Uniqueness Constraints:** Primary key or unique constraints are not defined for some tables. Identifier uniqueness should be verified to prevent duplicate records.

- **Data Consistency:** Free-text and categorical fields may contain inconsistent spelling, capitalization, formatting, or naming conventions.

- **Referential Integrity:** The Inventory table references Product and Branch records. These relationships should be validated to ensure that all foreign key values correspond to existing records.

- **Default Values:** Appropriate default values may be needed for selected fields, depending on business rules.

## 3. Issues Requiring Investigation

- **Business Rules:** Confirm which columns must contain values and which are legitimately allowed to be `NULL`.

- **Identifier Uniqueness:** Verify the uniqueness of identifiers such as `productid`, `customerid`, `supplierid`, and `branchid` before defining primary key constraints.

- **Foreign Key Relationships:** Confirm that every `productid` and `branchid` in the Inventory table references an existing record in the corresponding master table.

- **Data Types:** Investigate whether monetary values require decimal precision, whether measurement fields contain units, and whether the selected data types are appropriate for their intended uses.

- **Categorical Values:** Examine fields such as `customertype`, `category`, `industry_segment`, `region`, and `criticalitylevel` for inconsistent or unexpected values.

- **Business-Specific Fields:** Validate the expected formats and acceptable ranges for fields such as `chinataxid`, `importdutyrate`, `reliabilityscore`, and `warehousecapacity`.

- **Default Values:** Determine whether fields such as `current_stock`, `reorderlevel`, and `safetystock` require default values based on business requirements.

## 4. Recommendations

- **Improve Schema Design:** Review data types to ensure they are appropriate for identifiers, monetary values, percentages, dates, and measurements.

- **Enforce Data Integrity:** Define suitable primary keys, unique constraints, foreign keys, and `NOT NULL` constraints after validating existing data and confirming business requirements.

- **Standardize Categorical Data:** Apply consistent naming conventions and use controlled values or reference tables where appropriate.

- **Implement Data Validation:** Introduce validation rules for identifiers, tax IDs, numeric ranges, percentages, and other fields with specific business requirements.

- **Preserve Identifier Formats:** Use `VARCHAR` or `CHAR` for pincodes and similar identifiers where necessary to preserve leading zeros and formatting.

- **Improve Financial Precision:** Consider `DECIMAL` for monetary and percentage fields that require fractional values and accurate calculations.

- **Apply Appropriate Defaults:** Define default values only when they reflect valid business meanings. Avoid automatically replacing missing values with zero.

- **Establish Routine Data Quality Checks:** Regularly monitor missing values, duplicate records, inconsistent categories, invalid formats, and broken foreign key relationships.

- **Document Data Quality Rules:** Maintain documentation of expected data types, required fields, valid values, relationships, and business rules for each master table.

## 5. Conclusion

The profiling highlights opportunities to improve the reliability, consistency, and integrity of the master tables through better schema design, data validation, and constraint enforcement.

The identified concerns should be investigated before implementing changes to ensure that they align with business requirements. Addressing these areas will provide a more reliable foundation for reporting, analytics, and other data warehouse operations.

> **Note:** Schema-level concerns do not necessarily indicate actual data quality failures. Columns that allow `NULL` values do not necessarily contain missing values, and the absence of primary key constraints does not automatically mean duplicate identifiers exist. Missing values and duplicate records should be reported according to the results of the profiling checks.


In [13]:
con.close()